# Cross-encoder reranking v2 — TPU edition

Same experiment as `reranking_v2.ipynb`, rewritten so the heavy work actually runs on a
**Colab TPU** via PyTorch/XLA.

### What changed and why

The original notebook used `sentence_transformers.CrossEncoder.predict()`, which is a CUDA/CPU
path — on a TPU runtime it would silently fall back to the VM's CPU. Three changes make TPU real:

1. **Explicit XLA device.** Models are moved to `torch_xla` device; the notebook prints which
   backend it actually got (`tpu` / `cuda` / `cpu`) so the result is never ambiguous.
2. **Fixed input shapes.** XLA recompiles the graph for every new tensor shape, which would make
   variable-length padding catastrophically slow. Every batch is padded to exactly
   `(batch_size, max_length)`, so each model compiles **once**.
3. **Batched across queries.** Instead of 200 separate `.predict()` calls of 20 pairs each, all
   4000 (query, candidate) pairs for a field are scored in uniform batches — far better TPU
   utilisation.

Encoding and reranking are both hand-rolled on `AutoModel` / `AutoModelForSequenceClassification`
rather than going through `sentence-transformers`, purely so the device and padding are under
our control.

### Fallback behaviour

If `torch_xla` cannot be imported or no TPU is attached, the notebook **falls back to CUDA and
then CPU automatically** and says so in its output. It will always produce a complete run; it
just tells you honestly what hardware produced it.

### Data

Fetched directly from the public GitHub repo — **nothing to upload**. Note that
`corpus_v2.json` is committed in the repo as `data/corpus.json` (3054 chunks: 2974
`wikipedia_ar` + 80 `pilot_synthetic`), which is the output `01_corpus_builder.ipynb` describes.

### How to run

`Runtime → Run all`. Every model is checkpointed to Drive after it finishes, so an interruption
never costs completed work.

### 1. Install PyTorch/XLA (matched to the runtime's existing torch, so no restart is needed)

In [16]:
import importlib.util, subprocess, sys

# Install torch_xla pinned to the ALREADY-INSTALLED torch version. Installing an
# unpinned torch_xla would drag in a different torch and force a runtime restart,
# which would wipe session state mid-notebook.
if importlib.util.find_spec("torch_xla") is None:
    import torch as _t
    _v = _t.__version__.split("+")[0]
    print(f"torch {_v} present, torch_xla missing -> installing torch_xla=={_v}")
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"torch_xla[tpu]=={_v}",
         "-f", "https://storage.googleapis.com/libtpu-releases/index.html"],
        capture_output=True, text=True)
    print("pip exit", r.returncode)
    if r.returncode != 0:
        print(r.stderr[-2000:])
        print("\nInstall failed -- the notebook will fall back to CUDA/CPU and say so.")
else:
    print("torch_xla already available")

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "rank_bm25", "transformers", "sentencepiece"], check=False)
print("deps ready")

torch_xla already available
deps ready


### 2. Pick the device — and state plainly which one we actually got

In [17]:
import os, gc, json, re, pickle, time
import numpy as np
import pandas as pd
import torch

BACKEND = "cpu"
device = torch.device("cpu")
xm = None

try:
    import torch_xla
    import torch_xla.core.xla_model as _xm
    xm = _xm
    # torch_xla >= 2.5 prefers torch_xla.device(); older exposes xm.xla_device().
    device = torch_xla.device() if hasattr(torch_xla, "device") else xm.xla_device()
    _ = (torch.ones(2, 2, device=device) * 2).sum().item()   # force a real TPU op
    BACKEND = "tpu"
except Exception as e:
    print(f"XLA unavailable ({type(e).__name__}: {str(e)[:160]})")
    if torch.cuda.is_available():
        device, BACKEND = torch.device("cuda"), "cuda"
    else:
        device, BACKEND = torch.device("cpu"), "cpu"


def sync():
    """Flush the XLA graph. No-op off TPU."""
    if BACKEND == "tpu":
        if hasattr(torch_xla, "sync"):
            torch_xla.sync()
        else:
            xm.mark_step()


print("=" * 78)
print(f"BACKEND ACTUALLY IN USE: {BACKEND.upper()}   (device={device})")
if BACKEND == "tpu":
    print(f"torch {torch.__version__} | torch_xla {torch_xla.__version__}")
    print("All encoding and reranking below runs on the TPU.")
else:
    print("NOT running on TPU. Results are still valid, just slower.")
print("=" * 78)

BACKEND ACTUALLY IN USE: TPU   (device=xla:0)
torch 2.9.0+cpu | torch_xla 2.9.0
All encoding and reranking below runs on the TPU.


### 3. Config

In [18]:
CONFIG = {
    "corpus_url": "https://raw.githubusercontent.com/Rania-khaoudane/MSA/main/data/corpus.json",
    "wiki_qa_url": "https://raw.githubusercontent.com/Rania-khaoudane/MSA/main/data/qa_pairs_wiki.json",
    "base_encoder": "intfloat/multilingual-e5-base",
    "alpha": 0.8,

    # Candidates passed from retrieval into the reranker. Larger = higher
    # ceiling but slower, since the cross-encoder scores every candidate.
    "rerank_k": 20,

    "rerankers": [
        "BAAI/bge-reranker-v2-m3",                     # confirmed big win on T4
        "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",  # lighter multilingual baseline
        "BAAI/bge-reranker-base",                      # smaller BGE, size comparison
    ],

    # Fixed for XLA: every batch is padded to exactly these dims so the graph
    # compiles once per model instead of once per distinct shape.
    "max_length": 512,
    "batch_size": 32,
    "encode_batch_size": 32,

    "k_values": (1, 3, 5, 10),
    "bootstrap_n": 1000,
    "seed": 42,
}
CONFIG

{'corpus_url': 'https://raw.githubusercontent.com/Rania-khaoudane/MSA/main/data/corpus.json',
 'wiki_qa_url': 'https://raw.githubusercontent.com/Rania-khaoudane/MSA/main/data/qa_pairs_wiki.json',
 'base_encoder': 'intfloat/multilingual-e5-base',
 'alpha': 0.8,
 'rerank_k': 20,
 'rerankers': ['BAAI/bge-reranker-v2-m3',
  'cross-encoder/mmarco-mMiniLMv2-L12-H384-v1',
  'BAAI/bge-reranker-base'],
 'max_length': 512,
 'batch_size': 32,
 'encode_batch_size': 32,
 'k_values': (1, 3, 5, 10),
 'bootstrap_n': 1000,
 'seed': 42}

### 4. Load data straight from GitHub

In [19]:
import urllib.request

def fetch_json(url):
    with urllib.request.urlopen(url) as r:
        return json.loads(r.read().decode("utf-8"))

corpus = fetch_json(CONFIG["corpus_url"])
wiki_qa = fetch_json(CONFIG["wiki_qa_url"])

corpus_ids = [c["chunk_id"] for c in corpus]
corpus_texts = [c["text"] for c in corpus]
corpus_map = dict(zip(corpus_ids, corpus_texts))
known = set(corpus_ids)
qa = [q for q in wiki_qa if q["source_chunk_id"] in known]

# No training happens here, so the full benchmark is used as the test set.
print(f"Corpus {len(corpus)} | evaluating on all {len(qa)} Wikipedia items")

Corpus 3054 | evaluating on all 200 Wikipedia items


### 5. BM25 + hybrid retrieval (unchanged from earlier notebooks)

In [20]:
from rank_bm25 import BM25Okapi

DIAC = re.compile(r"[\u0610-\u061A\u064B-\u065F\u06D6-\u06DC\u06DF-\u06E8\u06EA-\u06ED\u0670]")

def normalize_arabic(t):
    t = DIAC.sub("", t)
    t = re.sub(r"[\u0625\u0623\u0622\u0627]", "\u0627", t)
    t = re.sub(r"\u0649", "\u064A", t); t = re.sub(r"\u0629", "\u0647", t)
    t = re.sub(r"\u0624", "\u0648", t); t = re.sub(r"\u0626", "\u064A", t)
    t = re.sub(r"\u0640+", "", t); t = re.sub(r"[^\w\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def tokenize(t):
    return normalize_arabic(t).split()

bm25 = BM25Okapi([tokenize(t) for t in corpus_texts])
_bm = {}
def bm25_scores(q):
    if q not in _bm:
        _bm[q] = np.asarray(bm25.get_scores(tokenize(q)))
    return _bm[q]

def minmax(a):
    lo, hi = a.min(), a.max()
    return (a - lo) / (hi - lo) if hi > lo else np.zeros_like(a)

print("BM25 index built")

BM25 index built


### 6. Fixed-shape batching — the part that makes TPU viable

Two helpers used by both stages. `batched_fixed` pads the final partial batch up to full size by
repeating its last element, then discards the padding from the results. Combined with
`padding="max_length"`, this guarantees every tensor entering the model has shape
`(batch_size, max_length)` — so XLA compiles one graph per model rather than one per shape.

In [21]:
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification

def batched_fixed(items, bs):
    """Yield (padded_batch, n_real). Last batch is padded up to bs for XLA."""
    for i in range(0, len(items), bs):
        chunk = list(items[i:i + bs])
        n = len(chunk)
        if n < bs:
            chunk += [chunk[-1]] * (bs - n)   # shape-stable padding, trimmed after
        yield chunk, n


def mean_pool(last_hidden, mask):
    m = mask.unsqueeze(-1).to(last_hidden.dtype)
    return (last_hidden * m).sum(1) / m.sum(1).clamp(min=1e-9)


@torch.no_grad()
def encode_texts(model, tok, texts, bs, max_len, label=""):
    """e5-style: mean pooling + L2 normalisation, on the selected device."""
    out, done, t0 = [], 0, time.time()
    for chunk, n in batched_fixed(texts, bs):
        enc = tok(chunk, padding="max_length", truncation=True,
                  max_length=max_len, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        h = model(**enc).last_hidden_state
        v = mean_pool(h, enc["attention_mask"])
        v = torch.nn.functional.normalize(v, p=2, dim=1)
        sync()
        out.append(v.float().cpu().numpy()[:n])
        done += n
        if done % (bs * 10) < bs:
            print(f"    {label} {done}/{len(texts)}  ({time.time() - t0:.0f}s)", flush=True)
    return np.concatenate(out, 0).astype("float32")


@torch.no_grad()
def score_pairs(model, tok, pairs, bs, max_len, label=""):
    """Cross-encoder relevance score for each (query, passage) pair."""
    out, done, t0 = [], 0, time.time()
    for chunk, n in batched_fixed(pairs, bs):
        enc = tok([a for a, _ in chunk], [b for _, b in chunk],
                  padding="max_length", truncation=True,
                  max_length=max_len, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        logits = model(**enc).logits
        sync()
        s = logits.float().cpu().numpy()
        # Rerankers emit a single relevance logit; classifiers emit 2 (take "relevant").
        s = s[:, 0] if s.shape[-1] == 1 else s[:, -1]
        out.extend(s[:n].tolist())
        done += n
        if done % (bs * 20) < bs:
            print(f"    {label} {done}/{len(pairs)}  ({time.time() - t0:.0f}s)", flush=True)
    return np.asarray(out)

print("helpers ready")

helpers ready


### 7. Stage 1 — retrieve candidates, then free the bi-encoder

In [22]:
print(f"Building index on {BACKEND.upper()}...")
t0 = time.time()

bi_tok = AutoTokenizer.from_pretrained(CONFIG["base_encoder"])
# Cast after loading rather than passing torch_dtype=: that kwarg is deprecated in
# newer transformers and absent in older ones, while .to(dtype=) works on every version.
bi = AutoModel.from_pretrained(CONFIG["base_encoder"]).to(device=device, dtype=torch.float32).eval()

corpus_emb = encode_texts(bi, bi_tok, [f"passage: {t}" for t in corpus_texts],
                          CONFIG["encode_batch_size"], CONFIG["max_length"], "corpus")
print(f"  corpus encoded in {time.time() - t0:.0f}s")

# Encode every query up front so the bi-encoder runs in uniform batches too.
q_fields = ["msa_query", "darija_query"]
query_emb = {}
for field in q_fields:
    query_emb[field] = encode_texts(bi, bi_tok, [f"query: {q[field]}" for q in qa],
                                    CONFIG["encode_batch_size"], CONFIG["max_length"], field)
    print(f"  {field} encoded")

K = CONFIG["rerank_k"]
candidates = {}
for field in q_fields:
    d = {}
    for i, q in enumerate(qa):
        s = (CONFIG["alpha"] * minmax(corpus_emb @ query_emb[field][i])
             + (1 - CONFIG["alpha"]) * minmax(bm25_scores(q[field])))
        d[q["id"]] = [corpus_ids[j] for j in np.argsort(-s)[:K]]
    candidates[field] = d
    print(f"  {field} candidates done")

del bi, corpus_emb, query_emb
gc.collect()
if BACKEND == "cuda":
    torch.cuda.empty_cache()
print(f"Stage 1 total {time.time() - t0:.0f}s")

Building index on TPU...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

    corpus 320/3054  (1s)
    corpus 640/3054  (2s)
    corpus 960/3054  (3s)
    corpus 1280/3054  (3s)
    corpus 1600/3054  (4s)
    corpus 1920/3054  (5s)
    corpus 2240/3054  (6s)
    corpus 2560/3054  (7s)
    corpus 2880/3054  (8s)
  corpus encoded in 11s
  msa_query encoded
  darija_query encoded
  msa_query candidates done
  darija_query candidates done
Stage 1 total 15s


### 8. The ceiling: what is the best a reranker could possibly do?

In [23]:
print("=" * 78)
print(f"CEILING ANALYSIS - Recall@{K} of the retrieval stage")
print("=" * 78)
print("A reranker can only reorder what retrieval already returned. Recall@K is")
print("therefore a hard upper bound on post-rerank Recall@1.\n")

ceiling = {}
for field in q_fields:
    hits = [int(q["source_chunk_id"] in candidates[field][q["id"]]) for q in qa]
    ceiling[field] = float(np.mean(hits))
    print(f"  {field:<14} Recall@{K} = {ceiling[field]:.3f}   <- reranking ceiling")

CEILING ANALYSIS - Recall@20 of the retrieval stage
A reranker can only reorder what retrieval already returned. Recall@K is
therefore a hard upper bound on post-rerank Recall@1.

  msa_query      Recall@20 = 1.000   <- reranking ceiling
  darija_query   Recall@20 = 0.965   <- reranking ceiling


### 9. Baseline (no reranking) for comparison

In [24]:
def evaluate_order(ordered_ids_by_qid):
    """Per-item hit vectors from an ordered candidate list, for bootstrapping."""
    out = {f"R@{k}": [] for k in CONFIG["k_values"]}
    rr = []
    for q in qa:
        ordered = ordered_ids_by_qid[q["id"]]
        gold = q["source_chunk_id"]
        pos = ordered.index(gold) + 1 if gold in ordered else None
        for k in CONFIG["k_values"]:
            out[f"R@{k}"].append(1.0 if (pos is not None and pos <= k) else 0.0)
        rr.append(1.0 / pos if pos else 0.0)
    return {**{k: np.array(v) for k, v in out.items()}, "MRR": np.array(rr)}

results = {}
for field in q_fields:
    results[("no_rerank", field)] = evaluate_order(candidates[field])

print("\nBaseline (retrieval order, no reranking):")
for field in q_fields:
    m = results[("no_rerank", field)]
    print(f"  {field:<14} R@1={m['R@1'].mean():.3f}  R@5={m['R@5'].mean():.3f}  MRR={m['MRR'].mean():.3f}")

print(f"\nDarija headroom available to reranking: "
      f"{ceiling['darija_query'] - results[('no_rerank','darija_query')]['R@1'].mean():+.3f}")


Baseline (retrieval order, no reranking):
  msa_query      R@1=0.760  R@5=0.950  MRR=0.839
  darija_query   R@1=0.625  R@5=0.890  MRR=0.741

Darija headroom available to reranking: +0.340


### 10. Stage 2 — cross-encoder reranking on the TPU

All 4000 (query, candidate) pairs per field go through the model in uniform batches, then get
regrouped per query for ordering. Each model is checkpointed on completion, and the whole
load+score body is guarded so one model's failure cannot cost the models that already finished.

In [25]:
RESULTS_CACHE = "rerank_results_partial.pkl"
if os.path.exists(RESULTS_CACHE):
    with open(RESULTS_CACHE, "rb") as f:
        results.update(pickle.load(f))
    print(f"Resumed {len(results)} cached result entries.")

def save_results():
    with open(RESULTS_CACHE, "wb") as f:
        pickle.dump(results, f)

def rerank_with(model_name):
    """Score every (query, candidate) pair jointly and reorder.

    The ENTIRE body is guarded, not just model loading: a device-specific failure
    inside the forward pass must not kill the run and lose earlier models' results.
    """
    try:
        tok = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        ce = AutoModelForSequenceClassification.from_pretrained(
            model_name, trust_remote_code=True).to(device=device, dtype=torch.float32).eval()
    except Exception as e:
        print(f"  SKIPPED (load failed) {model_name}: {type(e).__name__}: {str(e)[:200]}")
        return None

    out = {}
    try:
        for field in q_fields:
            qids = [q["id"] for q in qa]
            # Flatten to one long uniform stream; first batch pays the XLA compile.
            flat = [(q[field], corpus_map[c])
                    for q in qa for c in candidates[field][q["id"]]]
            t0 = time.time()
            scores = score_pairs(ce, tok, flat, CONFIG["batch_size"],
                                 CONFIG["max_length"], f"{field}")
            reordered = {}
            for i, qid in enumerate(qids):
                cands = candidates[field][qid]
                s = scores[i * K:(i + 1) * K]
                reordered[qid] = [cands[j] for j in np.argsort(-s)]
            out[field] = reordered
            print(f"    {field} reranked ({len(flat)} pairs, {time.time() - t0:.0f}s)")
    except Exception as e:
        print(f"  SKIPPED (inference failed) {model_name}: {type(e).__name__}: {str(e)[:200]}")
        del ce
        gc.collect()
        return None

    del ce
    gc.collect()
    if BACKEND == "cuda":
        torch.cuda.empty_cache()
    return out


for name in CONFIG["rerankers"]:
    short = name.split("/")[-1]
    if (short, "darija_query") in results:
        print(f"\n=== {short} === (already done, skipping)")
        continue
    print(f"\n=== {short} ===  on {BACKEND.upper()}")
    reordered = rerank_with(name)
    if reordered is None:
        continue
    for field in q_fields:
        results[(short, field)] = evaluate_order(reordered[field])
    save_results()
    m = results[(short, "darija_query")]
    print(f"  Darija R@1={m['R@1'].mean():.3f}  R@5={m['R@5'].mean():.3f}  MRR={m['MRR'].mean():.3f}")

Resumed 8 cached result entries.

=== bge-reranker-v2-m3 === (already done, skipping)

=== mmarco-mMiniLMv2-L12-H384-v1 === (already done, skipping)

=== bge-reranker-base === (already done, skipping)


### 11. Results table

In [26]:
rows = []
for (method, field), m in results.items():
    rows.append({"method": method, "query": field,
                 **{k: float(v.mean()) for k, v in m.items()}})
df = pd.DataFrame(rows)
df.to_csv("rerank_results.csv", index=False)

print("\n" + "=" * 78)
print(f"RESULTS   (backend: {BACKEND.upper()})")
print("=" * 78)
pivot = df.pivot(index="method", columns="query", values=["R@1", "R@5", "MRR"])
print(pivot.to_string(float_format=lambda x: f"{x:.3f}"))


RESULTS   (backend: TPU)
                                      R@1                    R@5                    MRR          
query                        darija_query msa_query darija_query msa_query darija_query msa_query
method                                                                                           
bge-reranker-base                   0.365     0.520        0.720     0.845        0.527     0.661
bge-reranker-v2-m3                  0.800     0.875        0.960     0.995        0.872     0.925
mmarco-mMiniLMv2-L12-H384-v1        0.715     0.865        0.925     0.970        0.810     0.913
no_rerank                           0.620     0.755        0.905     0.950        0.742     0.837


### 12. Improvement over no reranking, with paired bootstrap CIs

In [27]:
rng = np.random.default_rng(CONFIG["seed"])

def paired(a, b):
    d = np.asarray(a) - np.asarray(b)
    idx = rng.integers(0, len(d), size=(CONFIG["bootstrap_n"], len(d)))
    boot = d[idx].mean(axis=1)
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return d.mean(), lo, hi

print("\n" + "=" * 78)
print("RERANKING GAIN over retrieval order (paired 95% CI)")
print("=" * 78)
gain_rows = []
for method in df.method.unique():
    if method == "no_rerank":
        continue
    for field in q_fields:
        if (method, field) not in results:
            continue
        for metric in ["R@1", "MRR"]:
            d, lo, hi = paired(results[(method, field)][metric],
                               results[("no_rerank", field)][metric])
            sig = "yes" if lo > 0 else ("worse" if hi < 0 else "no")
            gain_rows.append({"method": method, "query": field, "metric": metric,
                              "gain": d, "lo": lo, "hi": hi, "significant": sig})
gains = pd.DataFrame(gain_rows)
print(gains.to_string(index=False, float_format=lambda x: f"{x:+.3f}"))
gains.to_csv("rerank_gains.csv", index=False)


RERANKING GAIN over retrieval order (paired 95% CI)
                      method        query metric   gain     lo     hi significant
          bge-reranker-v2-m3    msa_query    R@1 +0.120 +0.075 +0.170         yes
          bge-reranker-v2-m3    msa_query    MRR +0.088 +0.058 +0.120         yes
          bge-reranker-v2-m3 darija_query    R@1 +0.180 +0.120 +0.245         yes
          bge-reranker-v2-m3 darija_query    MRR +0.131 +0.094 +0.175         yes
mmarco-mMiniLMv2-L12-H384-v1    msa_query    R@1 +0.110 +0.060 +0.160         yes
mmarco-mMiniLMv2-L12-H384-v1    msa_query    MRR +0.076 +0.043 +0.110         yes
mmarco-mMiniLMv2-L12-H384-v1 darija_query    R@1 +0.095 +0.030 +0.165         yes
mmarco-mMiniLMv2-L12-H384-v1 darija_query    MRR +0.069 +0.027 +0.109         yes
           bge-reranker-base    msa_query    R@1 -0.235 -0.315 -0.155       worse
           bge-reranker-base    msa_query    MRR -0.177 -0.227 -0.120       worse
           bge-reranker-base darija_query    

### 13. Effect on the dialect gap, which is what the project is about

In [28]:
print("\n" + "=" * 78)
print("EFFECT ON THE DIALECT GAP (MSA - Darija)")
print("=" * 78)
gap_rows = []
for method in df.method.unique():
    if (method, "msa_query") not in results:
        continue
    for metric in ["R@1", "MRR"]:
        d, lo, hi = paired(results[(method, "msa_query")][metric],
                           results[(method, "darija_query")][metric])
        gap_rows.append({"method": method, "metric": metric, "gap": d,
                         "lo": lo, "hi": hi,
                         "gap_significant": "yes" if lo > 0 else "no"})
gapdf = pd.DataFrame(gap_rows).sort_values(["metric", "gap"])
print(gapdf.to_string(index=False, float_format=lambda x: f"{x:+.3f}"))
gapdf.to_csv("rerank_dialect_gap.csv", index=False)

print("""
Two outcomes are both worth reporting:
  - Gap SHRINKS  -> reranking mitigates dialect mismatch; a practical recommendation
                    that needs no training data at all.
  - Gap PERSISTS -> the penalty is not merely a ranking artefact of the bi-encoder;
                    it survives a much stronger ranking model, which is a stronger
                    claim about the phenomenon than anything measured so far.
""")


EFFECT ON THE DIALECT GAP (MSA - Darija)
                      method metric    gap     lo     hi gap_significant
          bge-reranker-v2-m3    MRR +0.053 +0.026 +0.081             yes
                   no_rerank    MRR +0.096 +0.065 +0.127             yes
mmarco-mMiniLMv2-L12-H384-v1    MRR +0.103 +0.069 +0.142             yes
           bge-reranker-base    MRR +0.133 +0.085 +0.179             yes
          bge-reranker-v2-m3    R@1 +0.075 +0.040 +0.120             yes
                   no_rerank    R@1 +0.135 +0.090 +0.180             yes
mmarco-mMiniLMv2-L12-H384-v1    R@1 +0.150 +0.095 +0.210             yes
           bge-reranker-base    R@1 +0.155 +0.080 +0.230             yes

Two outcomes are both worth reporting:
  - Gap SHRINKS  -> reranking mitigates dialect mismatch; a practical recommendation
                    that needs no training data at all.
  - Gap PERSISTS -> the penalty is not merely a ranking artefact of the bi-encoder;
                    it survives a mu

### 14. Headline

In [29]:
best = df[(df["query"] == "darija_query") & (df.method != "no_rerank")]
if len(best):
    top = best.loc[best["R@1"].idxmax()]
    base_r1 = df[(df.method == "no_rerank") & (df["query"] == "darija_query")]["R@1"].iloc[0]
    print("=" * 78)
    print(f"BEST RERANKER: {top['method']}")
    print("=" * 78)
    print(f"  Darija R@1   {base_r1:.3f} -> {top['R@1']:.3f}   ({top['R@1'] - base_r1:+.3f})")
    print(f"  Ceiling (Recall@{K})            {ceiling['darija_query']:.3f}")
    print(f"  Headroom captured              "
          f"{(top['R@1'] - base_r1) / max(ceiling['darija_query'] - base_r1, 1e-9) * 100:.1f}%")
    print(f"  Hardware                       {BACKEND.upper()}")
else:
    print("No reranker loaded successfully - check the model names in CONFIG.")

BEST RERANKER: bge-reranker-v2-m3
  Darija R@1   0.620 -> 0.800   (+0.180)
  Ceiling (Recall@20)            0.965
  Headroom captured              52.2%
  Hardware                       TPU


### 15. Persist the CSVs

Copies the three result CSVs next to this notebook in Drive. The executed notebook itself is
saved by Colab's own autosave, so `File → Save` (or just letting it autosave) writes this file
back to Drive **with all outputs included**.

In [30]:
import shutil, glob

OUT = None
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    OUT = "/content/drive/MyDrive/MSA_rerank_v2_tpu"
    os.makedirs(OUT, exist_ok=True)
except Exception as e:
    print(f"Drive not mounted ({type(e).__name__}); leaving CSVs in the working dir.")

if OUT:
    for f in glob.glob("rerank_*.csv") + glob.glob("rerank_results_partial.pkl"):
        shutil.copy(f, OUT)
        print("saved", os.path.join(OUT, os.path.basename(f)))

print(f"\nRun complete on {BACKEND.upper()}.")
print("Now use File -> Save, then tell Claude the run is done.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
saved /content/drive/MyDrive/MSA_rerank_v2_tpu/rerank_gains.csv
saved /content/drive/MyDrive/MSA_rerank_v2_tpu/rerank_dialect_gap.csv
saved /content/drive/MyDrive/MSA_rerank_v2_tpu/rerank_results.csv
saved /content/drive/MyDrive/MSA_rerank_v2_tpu/rerank_results_partial.pkl

Run complete on TPU.
Now use File -> Save, then tell Claude the run is done.
